# Gợi ý phim: cùng thể loại, khác góc nhìn
**Sau bài 01 · 2–3 giờ, làm cá nhân hoặc cặp.** Đây là thời gian dự án bổ sung, không cộng vào 150 phút của bài học.

**Sản phẩm:** Một hàm tìm top-k phim tương tự, bảng giải thích thể loại chung và biểu đồ điểm cosine.

Đọc → dự đoán → chạy mã mẫu → tự điền TODO → kiểm thử → giải thích. Các TODO cố ý dừng bằng `NotImplementedError`; đây không phải lỗi môi trường. Không thay kiểm thử bằng giá trị viết cứng. Giữ seed, Restart Kernel và chạy lại trước khi nộp.

Nguồn, điều kiện sử dụng và giới hạn dữ liệu: [bảng nguồn](../../course/DATA_SOURCES.md). Không tải mô hình pickle/joblib không rõ nguồn.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'mlcourse').is_dir()), None)
assert ROOT is not None, 'Mở notebook trong bản tải đầy đủ của repository.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from threadpoolctl import threadpool_limits
_threads = threadpool_limits(limits=2)
SEED = 42

OUT = ROOT/'outputs/projects/01_movie_recommender'
OUT.mkdir(parents=True, exist_ok=True)


## 1. Từ câu chuyện đến vector
Người dùng chọn một phim, hệ thống đưa ra phim gần về **thể loại**, không khẳng định người dùng sẽ thích. Mỗi trục là một thể loại; 1 nghĩa là có, 0 nghĩa là không. Đây là gợi ý dựa trên nội dung, không học từ rating.

Tính tay cosine của $u=(1,0,1)$ và $v=(1,1,1)$. Tại sao nhân cả vector với 3 không đổi cosine?


In [ ]:
def cosine(u, v):
    """Hai vector 1D cùng kích thước; trả float. Vector zero trả 0 theo quy ước ứng dụng."""
    u, v = np.asarray(u, float), np.asarray(v, float)
    if u.ndim != 1 or u.shape != v.shape: raise ValueError('Sai kích thước')
    # TODO: tích vô hướng / tích hai chuẩn; xử lý chuẩn bằng 0.
    raise NotImplementedError('Hoàn thiện cosine')


Gợi ý: `np.dot`, `np.linalg.norm`. Về toán học cosine với vector zero không xác định; giá trị 0 ở đây là quy ước để không đề xuất một mục không có metadata.


In [ ]:
assert np.isclose(cosine([1,0,1],[1,1,1]), 2/np.sqrt(6))
assert np.isclose(cosine([1,0],[3,0]), 1)
assert cosine([0,0],[1,0]) == 0
assert np.isclose(cosine([1,0],[-1,0]), -1)
print('Đã qua kiểm thử cosine')


## 2. Đọc dữ liệu thật và tạo vector — mã mẫu
Giữ ID phim để phân biệt các tên gần giống nhau. Không coi `(no genres listed)` là thể loại.


In [ ]:
movies = pd.read_csv(ROOT/'course/bai01/data/movies.csv')
assert movies.movieId.is_unique
movies = movies[movies.genres.ne('(no genres listed)')].reset_index(drop=True)
genre_sets = movies.genres.str.split('|').map(set)
genres = sorted(set.union(*genre_sets))
X = np.array([[int(g in tags) for g in genres] for tags in genre_sets], dtype=float)
display(movies.head())
print(X.shape, genres)


## 3. TODO: xếp hạng có thể giải thích
Nhận ID phim, trả DataFrame top-k có `movieId`, `title`, `score`. Không trả lại phim đầu vào; hòa điểm thì ưu tiên movieId nhỏ. Chỉ dùng cosine tự viết.


In [ ]:
def recommend(movie_id, k=5):
    # TODO: tìm vị trí ID, tính score; bỏ chính phim đó; sắp xếp ổn định.
    raise NotImplementedError('Hoàn thiện recommend')


Gợi ý: `movies.index[...]`, tạo cột `score`, lọc ID trước khi `head(k)`. Khi ID không tồn tại, báo `ValueError` rõ ràng.


In [ ]:
query_id = 1
top = recommend(query_id, 5)
assert len(top)==5 and query_id not in top.movieId.to_list()
assert top.score.between(0,1+1e-12).all()
assert top.score.is_monotonic_decreasing
assert top.movieId.is_unique
display(top)


## 4. Trực quan, xuất kết quả và thử một truy vấn mới
Điểm cosine không phải xác suất thích phim. Nhiều phim có cùng vector sẽ hòa điểm; thứ tự ID chỉ giúp tái lập, không phản ánh chất lượng.


In [ ]:
query_tags = genre_sets[movies.index[movies.movieId.eq(query_id)][0]]
lookup = dict(zip(movies.movieId, genre_sets))
top['shared_genres'] = top.movieId.map(lambda m: ', '.join(sorted(query_tags & lookup[m])))
top.to_csv(OUT/'recommendations.csv', index=False)
top.plot.barh(x='title', y='score', legend=False, figsize=(9,4), xlim=(0,1))
plt.tight_layout(); plt.savefig(OUT/'similarity.png'); plt.show()
display(recommend(260, 5))
(OUT/'model_card.json').write_text(json.dumps({'method':'genre cosine','query_id':query_id,'features':genres,'zero_policy':0,'limitation':'not personalized preference prediction'},ensure_ascii=False,indent=2),encoding='utf-8')


## 5. Kiểm tra sản phẩm
Chọn 3 phim khác nhau; đọc thể loại và giải thích vì sao top-1 đứng đầu. Thử phim khác tên nhưng cùng thể loại. Đề xuất cách bổ sung nội dung mà không đánh đồng tương đồng với mức độ yêu thích.


## Hoàn thiện sản phẩm và tự kiểm
Không báo accuracy khi chưa có nhãn phù hợp. Không gọi hệ này là gợi ý cá nhân hóa. Mở rộng tùy chọn: thêm trọng số thể loại và so sánh top-k, hoặc xây đánh giá bằng rating với cách chia dữ liệu riêng.

Nộp notebook đã chạy, thư mục đầu ra của dự án và `REPORT.md` tự viết: vấn đề → dữ liệu → phương pháp → kết quả → giới hạn. Dẫn nguồn, ghi seed/cách chia và mô tả ít nhất một trường hợp thất bại. Không chỉ chụp hình điểm số.

**Câu hỏi phản tư:** Điều gì trong kết quả là bằng chứng, điều gì mới là giả thuyết? Nếu dữ liệu mới khác dữ liệu học, bạn cần kiểm tra gì?

**Nhiệm vụ mở:** thay đổi một lựa chọn có lý do, xác định tiêu chí trước khi chạy và so sánh trên validation (hoặc phân tích độ ổn định với bài phân cụm). Không liên tục xem test để chọn cách làm.
